# Notebook 2 ? Drawdown-Penalised PPO on DOGEUSDT Replay

In [ ]:
import sys
import pathlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "procs").exists()), pathlib.Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline

from stable_baselines3 import PPO
from procs.stochastic_processes import MarketReplayMidpriceModel
from procs.gym.calibration import tune_gamma
from procs.gym.data_loader import load_single_day
from procs.gym.experiment_config import ReplayExperimentConfig
from procs.gym.helpers.plotting import plot_trajectory
from procs.gym.notebook_support import (
    build_replay_env,
    evaluate_agent_over_seeds,
    evaluate_as_fast,
    freeze_vecnorm,
    make_vecnorm,
    run_qmax_sensitivity,
    summarise_agent_frames,
)
from procs.gym.reward_scale import estimate_reward_scale
from procs.gym.sb3_wrapper import StableBaselinesTradingEnvironment
from procs.rewards import CjMmDrawdownPenalty, PnLReward
from procs.agents import AvellanedaStoikovAgent, Sb3Agent

## 1. Load Data and Shared Replay Configuration

In [ ]:
cfg = ReplayExperimentConfig()
cfg.ensure_artifact_dirs()

tb_log_dir = cfg.repo_root / "tb_logs_dd"
tb_log_dir.mkdir(parents=True, exist_ok=True)

DATA_PATH = cfg.data_path()
S, dt_sec, dt_index = load_single_day(str(DATA_PATH))
T_sec = float(dt_sec.sum())
sigma = MarketReplayMidpriceModel(S, dt_sec).volatility
print(f"Loaded {len(S):,} snapshots from {DATA_PATH.name}, ?={sigma:.6f}, T={T_sec:.0f}s")

## 2. Calibrate A-S Gamma, Reward Scale, and Inspect Q_MAX

In [ ]:
as_gamma, study = tune_gamma(
    midprices=S,
    dt_array=dt_sec,
    sigma=sigma,
    kappa=cfg.kappa,
    A=cfg.A,
    tick_size=cfg.tick_size,
    Q_MAX=cfg.q_max,
    gamma_range=cfg.as_gamma_range,
    n_trials=cfg.as_gamma_trials,
    num_trajectories=cfg.as_gamma_num_trajectories,
)
print(f"Using A-S ? = {as_gamma:.6f}")

reward_scale = estimate_reward_scale(
    midprices=S,
    dt_array=dt_sec,
    sigma=sigma,
    kappa=cfg.kappa,
    A=cfg.A,
    terminal_time=T_sec,
    tick_size=cfg.tick_size,
    Q_MAX=cfg.q_max,
    num_trajectories=cfg.reward_scale_num_trajectories,
    use_bm=False,
)
print(f"Reward scale: {reward_scale:.4f}")

qmax_sensitivity = run_qmax_sensitivity(
    S,
    dt_sec,
    gamma=as_gamma,
    sigma=sigma,
    kappa=cfg.kappa,
    A=cfg.A,
    terminal_time=T_sec,
    tick_size=cfg.tick_size,
    qmax_candidates=(10, 20, 50),
    num_trajectories=cfg.evaluation_rollouts,
    seed=cfg.evaluation_seed,
)
print("Q_MAX sensitivity (A-S baseline):")
print(qmax_sensitivity.to_string(index=False, float_format="%.6f"))

## 3. Build the Shared Replay Environment

In [ ]:
def get_doge_env(reward_fn=None):
    return build_replay_env(S, dt_sec, cfg, reward_fn=reward_fn)

## 4. A-S Baseline

In [ ]:
as_agent = AvellanedaStoikovAgent(as_gamma, sigma, cfg.kappa, T_sec, cfg.tick_size)
plot_trajectory(get_doge_env(PnLReward()), as_agent, seed=cfg.evaluation_seed, datetime_index=dt_index)

seed_range = range(cfg.evaluation_seed, cfg.evaluation_seed + cfg.evaluation_rollouts)
as_eval = evaluate_as_fast(
    S,
    dt_sec,
    gamma=as_gamma,
    sigma=sigma,
    kappa=cfg.kappa,
    A=cfg.A,
    terminal_time=T_sec,
    tick_size=cfg.tick_size,
    q_max=cfg.q_max,
    seeds=seed_range,
)
print(summarise_agent_frames({"A-S Baseline": as_eval}).to_string(float_format="%.6f"))

## 5. Train Drawdown-Penalised PPO

In [ ]:
%%time
train_env = get_doge_env(
    reward_fn=CjMmDrawdownPenalty(
        per_step_inventory_aversion=cfg.phi,
        drawdown_penalty=cfg.alpha_dd,
    )
)
sb3_train = make_vecnorm(StableBaselinesTradingEnvironment(train_env), cfg, training=True)

model_dd = PPO(
    "MlpPolicy",
    sb3_train,
    verbose=1,
    device="cpu",
    tensorboard_log=str(tb_log_dir),
    **cfg.ppo_kwargs(),
)
model_dd.learn(total_timesteps=cfg.ppo_total_timesteps)

dd_model_path = cfg.model_path("ppo_dd_penalised_doge")
dd_vecnorm_path = cfg.vecnorm_path("vecnorm_dd")
model_dd.save(str(dd_model_path))
sb3_train.save(str(dd_vecnorm_path))
print(f"Saved PPO model to {dd_model_path.with_suffix('.zip')}")
print(f"Saved VecNormalize stats to {dd_vecnorm_path}")

## 6. Evaluate on the Same Sample Shape

In [ ]:
eval_raw = StableBaselinesTradingEnvironment(get_doge_env(PnLReward()))
eval_vn = freeze_vecnorm(cfg.vecnorm_path("vecnorm_dd"), eval_raw, cfg)
ppo_dd_agent = Sb3Agent(model_dd, vecnorm_env=eval_vn)

ppo_eval = evaluate_agent_over_seeds(
    lambda: get_doge_env(PnLReward()),
    ppo_dd_agent,
    seeds=seed_range,
)
comparison = summarise_agent_frames({
    "A-S Baseline": as_eval,
    "DD-Penalised PPO": ppo_eval,
})
comparison.to_csv(cfg.result_path("nb2_dd_penalised_summary.csv"))
print(comparison.to_string(float_format="%.6f"))
print(f"Saved summary to {cfg.result_path('nb2_dd_penalised_summary.csv')}")

## 7. Artifacts

In [ ]:
print("Artifacts:")
print(f"  Model:   {cfg.model_path('ppo_dd_penalised_doge').with_suffix('.zip')}")
print(f"  VecNorm: {cfg.vecnorm_path('vecnorm_dd')}")
print(f"  Results: {cfg.result_path('nb2_dd_penalised_summary.csv')}")